In [ ]:
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# import seaborn as sns
# import warnings
# from keras.applications.resnet50 import ResNet50,preprocess_input
# from keras.preprocessing.image import load_img,img_to_array
# from keras.utils import plot_model,to_categorical
# from keras.applications.resnet50 import decode_predictions
# warnings.filterwarnings('ignore')

# res_net_model=ResNet50()

# img=load_img('/content/k.jpg',target_size=(224,224))
# img=img_to_array(img)
# img=np.expand_dims(img,axis=0)
# img=preprocess_input(img)
# res=res_net_model.predict(img)
# res2=decode_predictions(res)[0]
# res2

# from keras.models import Model
# feat_exctractor_model=Model(inputs=res_net_model.input,outputs=res_net_model.layers[-2].output)
# img_feat=feat_exctractor_model.predict(img)
# img_feat

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from keras.models import Model
from keras.layers import Dense,Flatten,Dropout,BatchNormalization,LSTM,Input
from keras.losses import categorical_crossentropy,sparse_categorical_crossentropy
from keras.optimizers import Adam
from keras.utils import plot_model,to_categorical
import keras
from pathlib import Path
import os



In [ ]:
import requests

url = "http://www.manythings.org/anki/ara-eng.zip"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36"
}

response = requests.get(url, headers=headers)
if response.status_code == 200:
    with open("ara-eng.zip", "wb") as file:
        file.write(response.content)
    print("File downloaded successfully.")
else:
    print(f"Failed to download file. Status code: {response.status_code}")

# !unzip /content/ara-eng.zip


In [ ]:
!unzip /content/ara-eng.zip

In [ ]:
with open("/content/ara.txt",'r') as f:
  text=f.read().split('\n')


ar_texts=[]
en_texts=[]
ar_vocab=set()
en_vocab=set()
num_sampels=10000

for i in range(min(len(text),num_sampels)):
  en_text,ar_text,_=text[i].split('\t')
  ar_text='\t'+ar_text+'\n'
  ar_texts.append(ar_text)
  en_texts.append(en_text)
  for ch in en_text:
    if ch not in en_vocab:
      en_vocab.add(ch)
  for ch in ar_text:
    if ch not in ar_vocab:
      ar_vocab.add(ch)

ar_vocab=sorted(list(ar_vocab))
en_vocab=sorted(list(en_vocab))
num_encoder_tokens=len(en_vocab)
num_decoder_tokens=len(ar_vocab)
max_encoder_seq_length=max([len(txt) for txt in en_texts])
max_decoder_seq_length=max([len(txt) for txt in ar_texts])

print("Number of samples:", len(ar_texts))
print("Number of unique input tokens:", num_encoder_tokens)
print("Number of unique output tokens:", num_decoder_tokens)
print("Max sequence length for inputs:", max_encoder_seq_length)
print("Max sequence length for outputs:", max_decoder_seq_length)

encoder_c2id={}
encoder_id2c={}
decoder_c2id={}
decoder_id2c={}

for c,id in enumerate(en_vocab):
  encoder_c2id[id]=c
  encoder_id2c[c]=id

for c,id in enumerate(ar_vocab):
  decoder_c2id[id]=c
  decoder_id2c[c]=id


In [ ]:
encoder_input_data = np.zeros(
    (len(en_texts), max_encoder_seq_length, num_encoder_tokens),
    dtype="float32",
)
decoder_input_data = np.zeros(
    (len(ar_texts), max_decoder_seq_length, num_decoder_tokens),
    dtype="float32",
)
decoder_target_data = np.zeros(
    (len(ar_texts), max_decoder_seq_length, num_decoder_tokens),
    dtype="float32",
)

for i, (input_text, target_text) in enumerate(zip(en_texts, ar_texts)):
    for t, char in enumerate(input_text):
        encoder_input_data[i, t, encoder_c2id[char]] = 1.0
    encoder_input_data[i, t + 1 :, encoder_c2id[" "]] = 1.0
    for t, char in enumerate(target_text):
        # decoder_target_data is ahead of decoder_input_data by one timestep
        decoder_input_data[i, t, decoder_c2id[char]] = 1.0
        if t > 0:
            # decoder_target_data will be ahead by one timestep
            # and will not include the start character.
            decoder_target_data[i, t - 1, decoder_c2id[char]] = 1.0
    decoder_input_data[i, t + 1 :, decoder_c2id[" "]] = 1.0
    decoder_target_data[i, t:, decoder_c2id[" "]] = 1.0

# the_word=""
# the_word2=""
# for i in decoder_input_data[0]:
#     the_word+=decoder_id2c[np.argmax(i)]
# for i in decoder_target_data[0]:
#     the_word2+=decoder_id2c[np.argmax(i)]

In [ ]:
latent_dim = 256

encoder_input=Input(shape=(None,num_encoder_tokens))
encoder_res,h_state,cell_state=LSTM(latent_dim,return_state=True,return_sequences=True)(encoder_input)

encoder_states=[h_state,cell_state]

decoder_input=Input(shape=(None,num_decoder_tokens))
decoder_res,_,_=LSTM(latent_dim,return_state=True,return_sequences=True)(decoder_input,initial_state=encoder_states)
decoder_dense=Dense(num_decoder_tokens,activation='softmax')
decoder_output=decoder_dense(decoder_res)

model=Model([encoder_input,decoder_input],decoder_output)

In [ ]:
# plot_model(model,show_shapes=True)
model.summary()

In [ ]:
model.compile(
    optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"]
)

model.fit(
    [encoder_input_data, decoder_input_data],
    decoder_target_data,
    batch_size=64,
    epochs=100,
    validation_split=0.2,
)

In [ ]:
# plt.plot(range(20),model.history.history['accuracy'])
# plt.plot(range(20),model.history.history['val_accuracy'])
# plt.plot(range(20),model.history.history['loss'])
# plt.plot(range(20),model.history.history['val_loss'])
# plt.legend(['accuracy','val_accuracy','loss','val_loss'])
# plt.show()

In [ ]:
keras.models.save_model(model,'/content/model.keras')

In [ ]:
model = keras.models.load_model('/content/model.keras')

In [ ]:
encoder_inputs = model.input[0]  # input_1
encoder_outputs, state_h_enc, state_c_enc = model.layers[2].output  # lstm_1
encoder_states = [state_h_enc, state_c_enc]
encoder_model = keras.Model(encoder_inputs, encoder_states)

decoder_inputs = model.input[1]  # input_2
decoder_state_input_h = keras.Input(shape=(latent_dim,))
decoder_state_input_c = keras.Input(shape=(latent_dim,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]
decoder_lstm = model.layers[3]
decoder_outputs, state_h_dec, state_c_dec = decoder_lstm(
    decoder_inputs, initial_state=decoder_states_inputs
)
decoder_states = [state_h_dec, state_c_dec]
decoder_dense = model.layers[4]
decoder_outputs = decoder_dense(decoder_outputs)
decoder_model = keras.Model(
    [decoder_inputs] + decoder_states_inputs, [decoder_outputs] + decoder_states
)

In [ ]:
def decode_sequence(input_seq):
    # Encode the input as state vectors.
    states_value = encoder_model.predict(input_seq, verbose=0)

    # Generate empty target sequence of length 1.
    target_seq = np.zeros((1, 1, num_decoder_tokens))
    # Populate the first character of target sequence with the start character.
    target_seq[0, 0, decoder_c2id["\t"]] = 1.0


    # Sampling loop for a batch of sequences
    # (to simplify, here we assume a batch of size 1).
    stop_condition = False
    decoded_sentence = ""
    while not stop_condition:
        output_tokens, h, c = decoder_model.predict(
            [target_seq] + states_value, verbose=0
        )

        # Sample a token
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_char = decoder_id2c[sampled_token_index]
        decoded_sentence += sampled_char

        # Exit condition: either hit max length
        # or find stop character.
        if sampled_char == "\n" or len(decoded_sentence) > max_decoder_seq_length:
            stop_condition = True

        # Update the target sequence (of length 1).
        target_seq = np.zeros((1, 1, num_decoder_tokens))
        target_seq[0, 0, sampled_token_index] = 1.0

        # Update states
        states_value = [h, c]
    return decoded_sentence

In [ ]:
for seq_index in range(20):
    # Take one sequence (part of the training set)
    # for trying out decoding.
    input_seq = encoder_input_data[seq_index : seq_index + 1]
    decoded_sentence = decode_sequence(input_seq)
    print("-")
    print("Input sentence:", en_texts[seq_index])
    print("Decoded sentence:", decoded_sentence)